# Guardián · Defensive Acceleration Hackathon 2026

Este notebook corre todo el proyecto desde el navegador: las pruebas, las facturas, la medición y los resultados.
**Cómo usarlo:** ejecuta las celdas de arriba hacia abajo (▶ o `Ctrl + Enter`). En Colab: *Entorno de ejecución → Ejecutar todas* sirve hasta la sección 5; desde la 6 se gastan llamadas de Gemini.

## 1. Traer el código

- **Opción A (más fácil):** al ejecutar la celda, sube el archivo `guardian_defacc.zip`.
- **Opción B:** guarda un token de GitHub en los Secretos de Colab (ícono de llave 🔑 a la izquierda) con el nombre `GITHUB_TOKEN`, y la celda clona el repositorio privado.

In [ ]:
import os, sys, zipfile, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if (Path.cwd() / "guardian").exists():                      # ya estamos dentro del repo (Jupyter local)
    RAIZ = Path.cwd()
else:
    RAIZ = Path("/content/guardian-defacc")
    if not RAIZ.exists():
        token = None
        if EN_COLAB:
            from google.colab import userdata
            try:
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        if token:
            subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/andresparada-dev/guardian-defacc", str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Sube guardian_defacc.zip:")
            subido = files.upload()
            with zipfile.ZipFile(next(iter(subido))) as z:
                z.extractall("/content")
            Path("/content/guardian_defacc").rename(RAIZ)
os.chdir(RAIZ)
print("Código en:", RAIZ)

## 2. Instalar

Instala las librerías y el OCR (Tesseract en español). Tarda 1 o 2 minutos.

In [ ]:
%%capture
!pip install -q -r requirements.txt
!apt-get install -y -q tesseract-ocr tesseract-ocr-spa

## 3. Guardar resultados en Google Drive (recomendado)

La caché de respuestas de Gemini evita pagar dos veces la misma llamada. En Colab se borra al cerrar la sesión, así que conviene guardarla en tu Drive. Si no quieres, salta esta celda.

In [ ]:
import shutil
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive/guardian_resultados")
    destino.mkdir(parents=True, exist_ok=True)
    local = RAIZ / "resultados"
    if local.exists() and not local.is_symlink():
        shutil.copytree(local, destino, dirs_exist_ok=True)
        shutil.rmtree(local)
    if not local.exists():
        local.symlink_to(destino)
    print("Resultados y caché en:", destino)
else:
    print("Fuera de Colab: los resultados quedan en la carpeta resultados/ del repo.")

## 4. Pruebas automáticas

Deben salir **18 passed**. Cada prueba es un ataque que la compuerta detiene, un límite conocido o una regla de cuota.

In [ ]:
!python -m pytest -q tests
!python verificar_entorno.py

## 5. Facturas y medición sin internet (peor caso simulado)

Genera 30 facturas sintéticas y corre todo con el **modelo comprometido**, que obedece todas las trampas. **No gasta llamadas.** Mide la compuerta, no a la IA.

In [ ]:
!python scripts/generar_facturas.py

from IPython.display import Image, display
for numero in ["FE-PRUEBA-0001", "FE-PRUEBA-0015", "FE-PRUEBA-0027"]:
    display(Image(f"datos/facturas/{numero}.png", width=420))

In [ ]:
!python scripts/medir.py --modelo comprometido --ocr simulado

In [ ]:
from IPython.display import Markdown
ultima = sorted((RAIZ / "resultados").glob("*_comprometido"))[-1]
display(Markdown((ultima / "resumen.md").read_text(encoding="utf-8")))
display(Image(str(ultima / "grafico_ataques.png")))

## 6. Clave de Gemini (desde aquí se gastan llamadas)

1. Saca tu clave gratis en https://aistudio.google.com/apikey
2. En Colab, guárdala en los Secretos (🔑) con el nombre `GEMINI_API_KEY` y activa el acceso del notebook.
3. Mira tu límite real en https://aistudio.google.com/rate-limit y escribe abajo el modelo con más llamadas por día que lea imágenes.

Los límites van por proyecto, no por clave, y se reinician a las 3:00 a.m. hora de Bogotá.

In [ ]:
MODELO = "gemini-3.5-flash-lite"   # gratis y lee imágenes; gemini-3.8-flash es más capaz. Revisa tus límites en AI Studio

clave = os.environ.get("GEMINI_API_KEY")
if EN_COLAB and not clave:
    from google.colab import userdata
    try:
        clave = userdata.get("GEMINI_API_KEY")
    except userdata.SecretNotFoundError:
        print("Falta el Secreto GEMINI_API_KEY: 🔑 (barra izquierda) → Agregar nuevo secreto → "
              "nombre GEMINI_API_KEY, pega la clave y activa 'Acceso del notebook'. Luego vuelve a correr esta celda.")
    except userdata.NotebookAccessError:
        print("El Secreto existe, pero falta activar 'Acceso del notebook' en 🔑. Actívalo y vuelve a correr esta celda.")
if clave:
    os.environ["GEMINI_API_KEY"] = clave
    os.environ["GEMINI_MODELO"] = MODELO
    !python verificar_entorno.py --probar-api gemini

### Prueba piloto (3 llamadas)

Antes de gastar 20 llamadas, prueba que todo conecta: 1 pregunta con trampa y 2 facturas con trampa. Si algo falla, copia el error completo y envíalo.

In [ ]:
!python scripts/medir.py --modelo gemini --preset minimo --casos A01 --escenarios A --max-llamadas 1
!python scripts/medir.py --modelo gemini --preset minimo --escenarios B --n-facturas 2 --max-llamadas 2

## 7. Mediciones reales

Según el panel de AI Studio de este proyecto, **Gemini 3.5 Flash Lite** y **Gemini 3.1 Flash Lite** dan 500 llamadas al día cada uno (15 por minuto), y **Gemini 3.8 Flash** da 20. La medición completa usa hasta unas 180 llamadas. Cada modelo tiene su propia cuota y su propia caché.

Si el panel de tu cuenta muestra menos, usa las corridas de 20 llamadas más abajo.

In [ ]:
# A · Medición completa con Gemini 3.5 Flash Lite (hasta unas 180 llamadas, ~14 minutos)
!python scripts/medir.py --modelo gemini:gemini-3.5-flash-lite --preset completo --espera 4.5 --max-llamadas 200

In [ ]:
# B · La misma medición con Gemini 3.1 Flash Lite, para comparar modelos (hasta unas 180 llamadas)
!python scripts/medir.py --modelo gemini:gemini-3.1-flash-lite --preset completo --espera 4.5 --max-llamadas 200

In [ ]:
# C · Muestra con el modelo más capaz, Gemini 3.8 Flash (20 llamadas: su cuota del día)
!python scripts/medir.py --modelo gemini:gemini-3.8-flash --preset minimo --n-ataques 5 --n-legitimas 3 --n-facturas 12 --espera 13 --max-llamadas 20

### Si tu cuota es de unas 20 llamadas al día
Corre estas tres, una por día o con claves de proyectos distintos (lo ya respondido sale de la caché):

In [ ]:
# Corrida 1 · sin defensa frente a solo la compuerta (20 llamadas)
!python scripts/medir.py --modelo gemini --preset minimo --n-ataques 5 --n-legitimas 3 --n-facturas 12 --max-llamadas 20

In [ ]:
# Corrida 2 · suma D1+D2 sobre los mismos casos (20 llamadas nuevas)
!python scripts/medir.py --modelo gemini --preset medio --n-ataques 5 --n-legitimas 3 --n-facturas 12 --max-llamadas 20

In [ ]:
# Corrida 3 · más casos (14 llamadas nuevas)
!python scripts/medir.py --modelo gemini --preset medio --n-ataques 10 --n-legitimas 5 --n-facturas 12 --max-llamadas 20

## 8. Ver los resultados

El resumen y el gráfico de la última corrida real, y una tabla con los ataques que lograron pasar (son los que van a la sección de límites de la ficha).

In [ ]:
import pandas as pd
reales = sorted(p for p in (RAIZ / "resultados").iterdir() if p.is_dir() and not p.name.endswith("_comprometido") and p.name != "cache_llm")
ultima = reales[-1]
display(Markdown((ultima / "resumen.md").read_text(encoding="utf-8")))
if (ultima / "grafico_ataques.png").exists():
    display(Image(str(ultima / "grafico_ataques.png")))
if (ultima / "detalle_cifras.csv").exists():
    a = pd.read_csv(ultima / "detalle_cifras.csv")
    display(a[(a.tipo == "ataque") & (~a.ok)][["config", "caso", "familia", "estado", "texto_final"]])
if (ultima / "detalle_facturas.csv").exists():
    b = pd.read_csv(ultima / "detalle_facturas.csv")
    display(b[b.variante.str.startswith("trampa") & (b.decision == "aprobar")][["config", "factura", "variante", "problemas"]])

In [ ]:
# Descargar la carpeta de la última corrida en un zip
import shutil
archivo = shutil.make_archive(f"/content/{ultima.name}", "zip", ultima)
if EN_COLAB:
    from google.colab import files
    files.download(archivo)
print(archivo)